# Agent와 Tool Calling

LLM은 학습한 지식으로 문장을 생성하지만 현재 날씨를 조회하거나 논문을 검색하는 외부 작업은 스스로 실행할 수 없다. 이 한계를 보완하려면 모델이 사용할 수 있는 외부 기능과 그 기능을 실행하는 제어 흐름이 필요하다.

- `Tool`: 검색, 계산, 데이터베이스 조회처럼 한 가지 작업을 수행하는 함수와 모델에 공개할 명세를 묶은 객체이다.
- `Tool Calling`: 모델이 Tool을 직접 실행하는 것이 아니라 호출할 Tool 이름과 인자를 구조화하여 요청하는 기능이다.
- `Agent`: 모델의 Tool 요청을 읽어 실제 Python 함수를 실행하고, 결과를 다시 모델에 전달하는 반복 실행 흐름이다.
- `Pydantic schema`: Tool 인자의 이름·자료형·기본값·제약을 선언하여 모델의 입력과 직접 호출 입력을 같은 기준으로 검증하는 구조이다.

따라서 모델은 **무엇을 호출할지 결정**하고, Agent 런타임은 **실제 함수를 실행**한다. Tool 결과가 충분하면 모델이 최종 답변을 만들고, 추가 정보가 필요하면 다른 Tool을 다시 선택할 수 있다.

## 현재 Agent 생성 경로

2026-08-09 기준 LangChain의 공식 Agents 문서는 `create_agent()`를 기본 생성 함수로 안내한다. `create_agent()`는 내부적으로 LangGraph의 그래프 기반 런타임을 만들고, 다음 두 단계를 필요한 만큼 반복한다.

- 모델 노드: 사용자 메시지와 Tool 명세를 읽고 답변 또는 `tool_calls`를 만든다.
- 도구 노드: 요청된 Python Tool을 실행하고 결과를 `ToolMessage`로 추가한다.

과거 예제의 `create_react_agent`, `AgentExecutor`, Hub ReAct 프롬프트 조립은 기존 자료를 읽을 때 참고할 수 있지만 수업의 기본 실행 경로로 사용하지 않는다. 수업에서는 현재 `create_agent()` 흐름으로 Wikipedia·arXiv·OpenWeatherMap·Tavily의 실제 데이터를 조회하며, 정적인 예시 데이터를 실제 조회 결과처럼 사용하지 않는다.

Tool은 모델의 지식을 자동으로 검증하지 않는다. Tool 설명과 입력 스키마가 모호하면 잘못된 Tool이나 인자가 선택될 수 있고, 외부 API 결과도 지연·오류·변경될 수 있다. 따라서 Tool 이름·인자·출처 URL과 중간 메시지를 함께 확인해야 한다.

## 공식 문서

- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [Tavily Search](https://docs.langchain.com/oss/python/integrations/tools/tavily_search)
- [OpenWeather Current Weather Data](https://openweathermap.org/current)


## 패키지 설치

`langchain`은 Agent 실행 흐름을, `langchain-openai`는 OpenAI Chat Model 연결을 제공한다.

- `pydantic`: Tool 인자의 이름·자료형·제약 조건을 정의한다.
- `wikipedia`: Wikipedia 검색과 문서 요약에 사용한다.
- `arxiv`: 논문 메타데이터와 초록 조회에 사용한다.
- `langchain-tavily`: 최신 웹 검색 결과를 Agent Tool로 제공한다.
- `requests`: OpenWeatherMap REST API를 직접 호출한다.
- `python-dotenv`: PyCharm 프로젝트의 `.env`를 읽는다.

설치 셀은 환경을 처음 구성할 때 한 번만 실행한다.


In [1]:
%pip install -U langchain langchain-openai langchain-tavily pydantic wikipedia arxiv requests python-dotenv

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl.metadata (3.6 kB)
Using cached lxml-6.1.1-cp312-cp312-win_amd64.whl (4.0 MB)
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=09559ae228f5d30a30ec9352cfab8c7b3dd31ad65994507c93047a20ec6843d9
  Stored in directory: c:\users\playdata2\appdata\local\pip\cache\wheels\63\47\7c\a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia

   ---------------------------------------- 0/5 [lxml]
   ---------------------------------------- 0/5 [lxml]
   ---------------------------------------- 0/5 [lxml]
   ---------------------------------------- 0/5 [lxml]
  


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 환경 변수와 모델 준비

프로젝트 루트의 `.env`에 저장한 키를 현재 Python 프로세스의 환경 변수로 등록한다. 키 값과 키의 설정 여부는 출력하지 않는다.

- `OPENAI_API_KEY`: Agent가 사용할 Chat Model 인증 키이다.
- `OPENAI_CHAT_MODEL`: Agent가 사용할 모델 이름이다.
- `OPENWEATHER_API_KEY`: 현재 날씨 조회용 키이다.
- `TAVILY_API_KEY`: 최신 웹 검색용 키이다.



In [2]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("프로젝트 폴더에 .env 파일을 생성한다.")

load_dotenv(dotenv_path=dotenv_path, override=False)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

CHAT_MODEL_NAME = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
if not OPENAI_API_KEY:
    raise RuntimeError(".env에 OPENAI_API_KEY를 설정한다.")

model = ChatOpenAI(
    model=CHAT_MODEL_NAME,
    temperature=0,
    use_responses_api=True,
)

## Tool 명세와 Agent 실행 순서

`@tool`은 Python 함수를 모델에게 보여 줄 Tool 객체로 변환한다.<br>
함수 코드 전체가 모델에게 전달되는 것은 아니다. 함수 이름, 설명과 입력 스키마가 Tool 명세로 전달되고 모델은 이 명세를 읽어 호출 여부와 인자를 결정한다.

- 함수 이름: 모델이 호출할 Tool을 식별하는 값이다.
- docstring: 모델이 Tool을 선택할 상황을 판단하는 설명이다.
- `args_schema`: Pydantic 모델을 LangChain의 JSON Schema로 바꾸어 인자의 이름·자료형·필수 여부·제약을 모델에 전달한다.
- 반환값: Tool 실행 후 모델이 읽고 최종 답변에 사용할 데이터이다.

이름이 비슷한 값은 쓰이는 위치가 다르다.

- `tools`: `create_agent()`에 넣는 **입력 목록**이다. 모델은 이 목록의 이름·설명·스키마를 보고 후보 Tool을 고른다.
- `tool_calls`: 모델이 만든 `AIMessage`의 **출력 목록**이다. 각 원소의 `name`, `args`, `id`는 실행할 함수와 인자, 호출 식별자이며 이 시점에는 함수가 아직 실행되지 않았다.
- `ToolMessage`: Python 함수가 실행한 결과를 다시 모델에 넣는 **입력 메시지**이다.
- 호출 ID(call ID): 한 번의 Tool 요청을 구분하는 식별자이다. `ToolMessage.tool_call_id`와 앞의 `tool_calls.id`가 같아야 병렬 또는 연속 호출에서도 어느 결과가 어느 요청에 대응하는지 연결할 수 있다.

Agent는 다음 흐름을 반복한다.

1. 사용자 메시지를 모델에 전달한다.
2. 모델이 `AIMessage.tool_calls`에 Tool 이름과 인자를 작성한다.
3. Agent 런타임이 `name`과 `args`로 해당 Python Tool을 실행한다.
4. 실행 결과를 같은 `tool_call_id`를 가진 `ToolMessage`로 모델에 전달한다.
5. 모델이 최종 `AIMessage`를 만들거나 추가 Tool을 요청한다.

핵심 흐름은 **사용자 메시지 → AIMessage.tool_calls → Python 함수 실행 → ToolMessage → 최종 AIMessage**이다. 최종 답변만 보면 실제 Tool 사용 여부를 알 수 없으므로 중간 메시지도 함께 확인해야 한다.


## 구조화된 인자를 받는 도구 만들기

생년월일과 기준일로 만 나이를 계산하는 Python 함수를 만든 뒤 Agent가 사용할 수 있는 Tool로 변환한다.

외부 API 없이 실행할 수 있으므로 `@tool`과 `args_schema`의 역할을 처음 확인하기 좋다.

### `@tool`: Python 함수를 Agent용 Tool로 변환한다

일반 Python 함수는 프로그램이 함수 이름과 인자를 직접 작성해야 실행된다. Agent가 함수를 선택하려면 함수의 이름, 사용 목적, 입력 형식을 모델이 읽을 수 있는 Tool 명세가 필요하다.

`@tool`은 바로 아래 함수를 LangChain의 Tool 객체로 바꾸는 데코레이터이다. 변환된 `calculate_age`에는 다음 정보와 실행 기능이 포함된다.

- Tool 이름: 함수 이름인 `calculate_age`이다.
- Tool 설명: 함수의 docstring인 ‘생년월일과 기준일을 받아 만 나이를 계산한다’이다.
- 입력 규칙: `args_schema`가 지정한 Pydantic 모델에서 가져온다.
- 실행 코드: Agent가 Tool을 선택하면 `calculate_age()` 함수 본문을 실행한다.

함수 정의 셀을 실행하는 순간 나이를 계산하는 것은 아니다. 이때는 Tool 객체만 만들어지고, 실제 계산은 아래의 `calculate_age.invoke(...)` 또는 이후 Agent 실행에서 시작된다.

### `args_schema=AgeInput`: Tool이 받을 입력 규칙을 지정한다

`args_schema`는 실제 생년월일 값을 넣는 자리가 아니다. **입력값의 이름·자료형·설명을 정의한 클래스**를 `@tool`에 알려 주는 인자이다.

`AgeInput`은 `BaseModel`을 상속한 Pydantic 입력 모델이다. 두 `Field`는 다음 규칙을 만든다.

- `birth_date: str`: 생년월일을 받을 필수 문자열이다.
- `as_of_date: str`: 나이를 계산할 기준일을 받을 필수 문자열이다.
- `description`: 각 값의 의미와 형식을 모델에게 알려 주는 Tool 명세이다.

`AgeInput`의 필드 이름과 `calculate_age()`의 매개변수 이름은 서로 일치해야 한다. LangChain은 입력 dict를 Pydantic 규칙으로 검사한 뒤, 각 값을 같은 이름의 함수 인자로 전달한다.

```text
{birth_date, as_of_date}
→ AgeInput으로 필드 이름과 자료형 검사
→ calculate_age(birth_date=..., as_of_date=...) 실행
→ {birth_date, as_of_date, age} 반환
```

따라서 `@tool`은 **함수를 Agent용 도구로 만드는 역할**, `args_schema`는 **그 도구에 전달할 입력 규칙을 정하는 역할**을 담당한다.


In [7]:
from datetime import date

from langchain.tools import tool
from pydantic import BaseModel, Field

# 1. Tool이 받을 두 날짜의 이름, 형식과 설명을 정한다.
class AgeInput(BaseModel):
    birth_date: str = Field(description='생년월일. YYYY-MM-DD 형식')

    # 기준 날짜
    as_to_date: str = Field(description='나이를 계산할 기준일. YYYY-MM-DD 형식')


# 2. @tool이 붙은 함수 정의
# -> Agent가 선택할 수 있는 Tool로 지정됨
# + Agent가 해당 함수를 사용할 수 있도록 명세(AgeInput) 등록
#  -> LLM이 명세를 보고서 같은 이름으로 값을 전달하기 때문에
#     함수의 매개 변수명도 명세에 작성된 이름과 같아야한다!
@tool(args_schema=AgeInput)
def calculate_age(birth_date: str, as_to_date: str) -> dict:
    """생년월일과 기준일을 받아 만 나이를 계산한다."""

    # str -> date 객체로 변환
    # -> 같은 날짜타입의 객체는 날짜 연산(+,-, 크기비교) 가능
    birth = date.fromisoformat(birth_date)
    target = date.fromisoformat(as_to_date)

    # 만 나이 계산
    # 연도 차이 - (생일 지났으면 1, 아니면 0)
    age = target.year - birth.year - (
        (target.month, target.day) < (birth.month, birth.day)
    )

    # 3. 입력 받은 두 날짜와 계산한 나이를 dict 형태로 반환
    return {
        "birth_date": birth.isoformat(),
        "as_to_date": target.isoformat(),
        "age": age,
    }

# 4. Tool이 정상 동작 하는지 확인
age_result = calculate_age.invoke({
    "birth_date": "1990-01-16",
    "as_to_date": "2027-01-01",
})

print(age_result)


{'birth_date': '1990-01-16', 'as_to_date': '2027-01-01', 'age': 36}


## 실제 Wikipedia 검색 도구

`wikipedia` 패키지는 Wikipedia 검색, 문서 열기와 요약 기능을 Python으로 제공한다. 먼저 Tool이 받을 검색어·언어·문장 수를 Pydantic 스키마로 정의한다.

- `Literal["ko", "en"]`: `language`가 한국어판 또는 영어판 코드만 받도록 제한한다.
- `Field(default=3, ge=1, le=5)`: 문장 수의 기본값을 3으로 정하고 1 이상 5 이하만 허용한다.
- `description`: Agent가 `sentences`를 요약 문장 수로 이해하도록 Tool 스키마에 설명을 넣는다.

이 제약은 모델이 잘못된 인자를 만들었을 때 실제 Wikipedia 요청 전에 입력 오류를 발견하게 한다.


In [9]:
from typing import Literal
import wikipedia

# wikipedia_api는 패키지가 실제 요청에 사용하는 API 주소를 담고 있음
from wikipedia import wikipedia as wikipedia_api

# User-Agent는 Wikipedia에 요청을 보낸 사용자 또는 에이전트가 누구인지 알려줌
wikipedia.set_user_agent(
    "SKN33-Lecture-BDH/1.0 (https://github.com/20260528-skn-family-ai-33)"
)

class WikipediaInput(BaseModel):
    """Wikipedia에서 찾을 사람 또는 기술, 사건의 이름을 입력 받는다"""
    query:str = Field(
        description="Wikipedia에서 찾을 인물 또는 기술, 사건"
    )

    language: Literal["ko", "en"] = Field(
        default="ko",
        description="검색에 사용할 Wikipedia 언어 코드"
    )

    sentences: int = Field(
        default=3,
        ge=1,
        le=5,
        description="요약에 포함할 문장 수"
    )

### Wikipedia 검색 Tool 구성하기

이 단계에서는 **입력 규칙**과 **실제 검색 함수**를 하나의 Tool로 연결한다.

#### 구성 요소

- `WikipediaInput`: Tool이 받을 `query`, `language`, `sentences`의 규칙을 정의한다.
- `@tool`: 바로 아래의 Python 함수를 LangChain Tool 객체로 변환한다.
- `args_schema=WikipediaInput`: Tool의 입력 검사에 `WikipediaInput` 규칙을 사용하도록 연결한다.
- `search_wikipedia()`: 검사를 통과한 인자로 Wikipedia를 실제 검색한다.

`args_schema`에는 검색값을 직접 넣지 않는다. 입력 규칙을 정의한 클래스 이름만 전달한다.

#### 실행 순서

1. Agent가 `query`, `language`, `sentences`를 가진 dict를 만든다.
2. `WikipediaInput`이 필드 이름, 자료형과 허용 범위를 검사한다.
3. 검사를 통과한 값이 같은 이름의 함수 매개변수로 전달된다.
4. `search_wikipedia()`가 문서를 검색하고 결과 dict를 반환한다.

```text
{"query": "Geenius", "language": "en", "sentences": 2}
→ WikipediaInput 검사
→ search_wikipedia(query="Geenius", language="en", sentences=2)
→ 제목·요약·URL이 담긴 dict
```

`language="fr"` 또는 `sentences=10`처럼 스키마 규칙을 벗어난 값은 Wikipedia에 요청하기 전에 오류가 발생한다.

#### 네트워크 호출과 반환값

- 함수 정의 셀 실행: Tool 객체만 만들며 Wikipedia에 요청하지 않는다.
- `.invoke()` 실행 또는 Agent의 Tool 선택: 실제 Wikipedia 요청이 발생한다.
- 검색 성공: `query`, `title`, `summary`, `url`을 반환한다.
- 검색 결과 없음: `query`, `error`를 반환한다.
- 동음이의어: `query`, `error`, `options`를 반환한다.

`DisambiguationError`는 같은 이름의 후보가 여러 개인 경우이고, `PageError`는 선택한 문서를 불러오지 못한 경우이다.


In [11]:
@tool(args_schema=WikipediaInput)
def search_wikipedia(
        query: str,
        language: str = "ko",
        sentences: int = 3
) -> dict:
    """Wikipedia에서 문서를 검색하고 제목, 요약, 원문 URL을 반환한다"""
    # docstring

    # 1. 검색할 Wikipedia 언어판을 선택한다
    wikipedia.set_lang(language)

    # 선택한 언어판의 Wikipedia API를 요청할 수 있도록 주소 지정
    wikipedia_api.API_URL = f'https://{language}.wikipedia.org/w/api.php'

    # 2. query를 검색하고 관련 제목을 최대 5개까지 가져온다.
    titles = wikipedia.search(query, results=5)

    # 검색된 제목이 하나도 없으면 함수 실행을 끝내고 실패 사유 반환
    if not titles:
        return {"query":query, "error":"검색 결과가 없습니다."}

    try:
        # 3. 검색 결과의 첫 번째 제목으로 실제 문서 열기
        # auto_suggest=False
        # -> 패키지가 제목을 다른 단어로 자동 변경하지 않게한다.
        page = wikipedia.page(titles[0], auto_suggest=False)

        # 4. 연 문서에서 sentences에 지정된 문장 수 만큼 요약을 가져옴
        summary = wikipedia.summary(
            page.title,
            sentences=sentences,
            auto_suggest=False,
        )

    except wikipedia.DisambiguationError as e:
        # 같은 이름의 문서가 여러 개이면, 사용자가 다시 고를 수 있도록 후보 5개를 반환
        return{
            "query": query,
            "error": "같은 이름의 문서가 여러 개 있다.",
            "options": e.options[:5],
        }

    except wikipedia.PageError as e:
        # 제목은 검색됐지만 해당 문서를 열 수 없는 경우
        return{
            "query": query,
            "error": "문서를 불러오지 못했습니다."
        }

    # 5. 검색에 성공하면 Agent가 답변에 활용할
    # 제목, 요약, 출처 URL을 dict 형태로 반환
    return {
        "query": query,
        "title": page.title,
        "summary": summary,
        "url": page.url,
    }


### Wikipedia Tool을 직접 호출하기

Agent에 연결하기 전에 `invoke()`로 실제 검색을 실행한다. 정상 조회에서는 제목·요약·URL을 확인하고, 문서가 없거나 여러 뜻을 가리키면 `error`와 후보 목록을 확인한다. 이 단계가 성공해야 이후 Agent 오류와 Wikipedia 조회 오류를 구분할 수 있다.


In [16]:
wikipedia_result = search_wikipedia.invoke(
    {
        "query": "Agent",
        "language": "ko",
        "sentences": 3,
    }
)

print(wikipedia_result)

{'query': 'Agent', 'error': '같은 이름의 문서가 여러 개 있다.', 'options': ['탤런트 에이전트', '탤런트 에이전트', '스포츠 에이전트', '지능형 에이전트', '사용자 에이전트']}


C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


## Agent에 Wikipedia Tool 연결하기

`create_agent()`는 Chat Model과 Tool 목록을 결합해 Tool Calling 반복을 수행하는 그래프를 만든다.

- `model`: 사용자 요청을 해석하고 Tool 호출 여부를 결정한다.
- `tools`: 모델이 선택할 수 있는 Tool 객체 목록이다.
- `system_prompt`: Tool을 사용할 조건과 최종 답변 형식을 정한다.
- 반환값: `invoke()`가 끝난 뒤 사용자 메시지부터 최종 답변까지 담은 `messages` 상태이다.

Agent 생성 자체는 모델이나 Tool을 실행하지 않는다. 실제 호출은 뒤의 `wikipedia_agent.invoke()`에서 일어난다.


In [17]:
from langchain.agents import create_agent

wikipedia_agent = create_agent(
    model = model, # 질문 해석과 Tool 호출 결정을 담당하는 ChatModel
    tools = [search_wikipedia], # Model에 공개할 실제 Tool 목록
    system_prompt=( # Tool 사용 지침, 행동 기준
        "사실 확인이 필요한 질문은 Wikipedia 도구를 사용한다."
        "도구 결과에 URL이 있으면 답변 마지막에 출처로 표시한다."
    )
)

### Agent 메시지에서 Tool 호출 흐름 읽기

Agent의 반환 상태에는 `HumanMessage`, `AIMessage`, `ToolMessage`가 시간 순서대로 저장된다.

`show_agent_trace()`는 이 목록에서 다음 연결을 찾아 사람이 읽을 수 있게 출력한다.

- `AIMessage.tool_calls`: 모델이 요청한 Tool 이름·인자·호출 ID이다.
- `ToolMessage.tool_call_id`: 실행 결과가 어느 요청에 대응하는지 나타낸다.
- 마지막 `AIMessage.text`: Tool 결과를 바탕으로 만든 사용자용 답변이다.

호출 ID(call ID)는 모델이 각 Tool 요청에 붙이는 고유 식별자이다. 코드에서 새로 만들거나 바꾸지 않고, `tool_calls.id`와 `ToolMessage.tool_call_id`를 그대로 비교해야 요청과 결과의 연결을 검증할 수 있다.


In [ ]:
from langchain.messages import AIMessage, ToolMessage

### Wikipedia Agent 실행 확인

사용자 질문은 `messages` 목록의 `role="user"`와 `content`로 전달한다. 반환 상태를 `show_agent_trace()`에 넣어 모델의 Tool 요청, Wikipedia 결과와 최종 답변이 순서대로 이어지는지 확인한다.

이 셀은 OpenAI API와 Wikipedia 네트워크를 실제로 사용한다. 최종 문장만 보지 않고 `search_wikipedia`의 인자와 반환 URL이 중간 메시지에 나타나는지 확인해야 한다.


## 실제 arXiv 논문 조회 도구

arXiv Tool은 논문 ID로 실제 메타데이터를 조회한다. 검색어가 아니라 Transformer 논문 ID인 `1706.03762`를 사용하므로 조회 대상을 재현하기 쉽다.

`ArxivInput`은 모델이 전달할 `paper_id` 필드를 정의한다. Tool은 이를 `arxiv.Search`의 `id_list`에 넣고 제목, 저자, 게시일, 초록과 URL만 dict로 반환한다. URL은 Agent가 답변의 출처로 그대로 사용한다.


In [ ]:
import arxiv

### arXiv Tool을 직접 호출하기

논문 ID를 Tool 입력 스키마와 같은 key로 전달한다. 정상 실행에서는 제목·저자·게시일·초록·URL을 확인하며, 결과가 없으면 `error` 필드를 확인한다. 실제 arXiv 네트워크 요청이 발생한다.


### 논문 정보를 사용하는 Agent

Wikipedia, arXiv, 나이 계산 Tool을 하나의 Agent에 등록한다. `tools`는 모델이 선택할 후보 목록일 뿐 모든 Tool이 매번 실행된다는 뜻은 아니다. 질문에 논문 ID가 포함되면 모델이 `search_arxiv`를 선택하고, 반환된 URL을 최종 답변의 근거로 사용하도록 `system_prompt`를 정한다.


In [ ]:
research_system_prompt = (
    "사실이나 논문 정보는 제공된 도구로 확인한다. "
    "계산이 필요하면 계산 도구를 사용한다. "
    "도구 결과에 URL이 있으면 출처로 표시한다."
)

research_user_prompt = (
    "arXiv 1706.03762 논문의 제목과 핵심 내용을 "
    "두 문장으로 설명하고 출처 URL을 알려 줘."
)

### 두 도구를 순서대로 사용하기

한 질문에 조회와 계산이 함께 필요하면 Agent가 Tool을 연속해서 호출할 수 있다. Wikipedia에서 찾은 생년월일이 `calculate_age`의 `birth_date`로 전달되는지 메시지 흐름을 확인한다.

`date.today()`는 코드 실행일의 로컬 날짜를 만들고 `isoformat()`은 Tool 스키마와 같은 `YYYY-MM-DD` 문자열로 바꾼다. 따라서 결과의 나이는 실행 날짜에 따라 달라진다.


## 실제 OpenWeatherMap 날씨 도구

OpenWeatherMap은 도시 이름을 받아 현재 날씨를 반환하는 REST API이다. REST API는 URL과 HTTP 메서드로 서버 자원을 요청하는 방식이며, 여기서는 데이터를 읽는 `GET` 요청을 사용한다. `requests`는 이 HTTP 요청을 Python에서 전송하는 라이브러리이다.

먼저 도시 이름과 온도 단위를 Pydantic 스키마로 정의한다.

- `city`: OpenWeatherMap이 검색할 영문 도시 이름이다.
- `units`: `metric`은 섭씨, `imperial`은 화씨를 반환한다.
- `Literal`: 두 단위 문자열 외의 값을 요청 전에 거부한다.


In [ ]:
import requests

### HTTP 응답을 날씨 dict로 정규화하기

`requests.get()`은 URL과 query parameter를 조합해 HTTP GET 요청을 보낸다. 서버는 상태 코드와 JSON 본문을 반환한다.

- `params`: `q`, `appid`, `units`, `lang`을 URL query parameter로 인코딩한다.
- `timeout=10`: 서버 응답을 최대 10초까지 기다린다.
- `response.ok`: 상태 코드가 성공 범위인지 확인한다.
- `response.json()`: 성공 응답의 JSON 문자열을 Python 딕셔너리와 목록으로 역직렬화한다.

`appid`는 query parameter이므로 완성된 요청 URL에는 API 키가 포함된다. 실패 응답은 URL이나 본문을 예외에 넣지 않고 상태 코드만 알린 뒤 중단하며, JSON 변환은 성공 응답에서만 수행한다.

Tool은 큰 API 응답에서 최종 답변에 필요한 도시·날씨·온도·체감 온도·습도만 선택한다. API 키는 요청에 사용하지만 출력이나 반환 dict에 포함하지 않는다.


### OpenWeatherMap Tool을 직접 호출하기

Agent 연결 전에 서울의 현재 날씨를 직접 조회한다. 정상 응답에서는 도시·설명·온도·체감 온도·습도·단위를 확인한다. 실패하면 HTTP 상태를 기준으로 인증 키, 도시 이름, 호출 한도와 서버 상태를 먼저 점검한다.


### 날씨 Agent 실행 확인

날씨 Tool을 Agent에 하나만 연결하고 사용자 질문을 전달한다. 모델은 질문에서 도시와 단위를 만들고, Tool 반환 dict를 사용자용 문장으로 바꾼다.

`create_agent()`의 `tools`는 후보 목록이고 `invoke()`의 `messages`는 실제 사용자 입력이다. 두 값을 구분해 확인한다.


## Tavily 최신 웹 검색 도구

Wikipedia와 arXiv는 특정 지식원에 적합하고, Tavily는 최신 웹 정보를 여러 출처에서 찾는 AI 검색 서비스이다. 일반 사용자가 검색 결과 페이지를 보는 포털보다는 개발자가 LLM이나 Agent에 웹 검색 기능을 연결하는 플랫폼에 가깝다.

- `Tavily Platform`: API Key를 발급하고 사용량을 확인하는 개발자 사이트이다.
- `Tavily Search API`: 검색어를 받아 `answer`, `title`, `url`, `content`와 같은 구조화된 결과를 반환한다.
- `TavilySearch`: Tavily Search API를 LangChain Tool로 사용할 수 있게 감싼 객체이다.

검색 흐름은 **사용자 질문 → Agent가 Tavily Tool 선택 → 웹 검색 → URL·본문 반환 → LLM 답변 생성** 순서이다.

### Tavily API Key 발급과 등록

1. [Tavily Platform](https://app.tavily.com/home)에 접속해 로그인하거나 계정을 생성한다.
2. Dashboard에 표시된 API Key를 복사한다.
3. 프로젝트의 `.env` 파일에 `TAVILY_API_KEY=복사한_API_KEY` 형식으로 저장한다.
4. 이미 Python 커널을 실행 중이었다면 커널을 다시 시작하거나 앞의 환경 변수 로딩 셀을 다시 실행한다.

API Key는 코드셀에 직접 작성하거나 화면에 출력하지 않으며 Git 저장소에도 올리지 않는다. 자세한 발급 흐름은 [Tavily 공식 Quickstart](https://docs.tavily.com/documentation/quickstart)에서 확인할 수 있다.

- `max_results`: 한 번의 검색에서 반환할 결과 수이다.
- `topic`: `general`, `news`, `finance` 중 검색 범위를 정한다.
- `search_depth`: `basic` 또는 `advanced`로 속도와 탐색 깊이를 조절한다.
- `include_answer`: Tavily가 검색 결과를 바탕으로 만든 요약 답변을 함께 받을지 정한다.
- `query`: `invoke()` 시점에 전달할 실제 검색 문장이다.

생성자의 검색 옵션은 직접 호출의 기본 동작을 정하고, `invoke()`의 `query`가 검색 대상을 정한다. Agent에 연결하면 모델이 Tool 스키마에 공개된 검색 옵션도 호출 인자로 선택할 수 있으므로 추적 출력에서 실제 `query`와 옵션을 함께 확인한다. 최신 정보 질문은 모델의 학습 지식만 사용하지 않고 개별 URL과 본문 근거를 확인한다.


In [ ]:
from langchain_tavily import TavilySearch

### 웹 검색 Agent 실행 확인

최신 정보 질문을 Tavily가 연결된 Agent에 전달한다. 모델이 `query`를 만들고 검색 결과에서 공식 문서 URL을 골라 최종 답변에 포함하는지 확인한다.

Tavily의 요약 `answer`만 신뢰하지 않고, 개별 결과의 `url`과 `content`가 질문의 근거인지 함께 확인한다.


In [ ]:
web_search_system_prompt = (
    "최신 정보 질문은 웹 검색 도구로 확인한다. "
    "답변에는 확인한 출처 URL을 함께 표시한다."
)

web_search_user_prompt = (
    "LangChain Python의 현재 권장 Agent 생성 함수가 무엇인지 "
    "공식 문서를 찾아 설명해 줘."
)

## 여러 도구를 사용하는 Agent

도구가 많아질수록 Tool의 이름과 설명이 겹치지 않아야 한다. 모델은 질문과 각 Tool 설명을 비교해 검색 범위가 가장 적절한 도구를 고른다.

- 논문 ID와 초록: `search_arxiv`를 사용한다.
- 인물·개념의 백과사전 정보: `search_wikipedia`를 사용한다.
- 현재 날씨: `get_current_weather`를 사용한다.
- 최신 웹 정보: `tavily_search`를 사용한다.
- 날짜 기반 만 나이: `calculate_age`를 사용한다.

한 질문에 서로 다른 정보가 필요하면 Agent가 도구를 순서대로 또는 모델이 지원하는 경우 병렬로 호출할 수 있다. 각 결과가 같은 질문의 어떤 하위 작업에 대응하는지 호출 인자와 ID로 확인한다.


In [ ]:
multi_tool_system_prompt = (
    "질문에 맞는 실제 조회 도구를 사용한다. "
    "도구가 반환한 값만 근거로 답하며 URL이 있으면 출처로 표시한다."
)

multi_tool_user_prompt = (
    "arXiv 1706.03762 논문의 제목과 오늘 서울 날씨를 함께 알려 줘. "
    "논문 출처 URL도 표시해 줘."
)

## 직접 적용하기

다음 요구사항을 만족하도록 도구와 Agent를 확장한다.

1. `convert_temperature` Tool을 만든다.
2. 입력은 `temperature`와 `from_unit`, `to_unit`으로 구성한다.
3. 날씨 조회 결과가 섭씨이면 화씨로, 화씨이면 섭씨로 변환한다.
4. `get_current_weather`와 새 Tool을 하나의 Agent에 등록한다.
5. “서울의 현재 온도를 섭씨와 화씨로 모두 알려 줘.”라고 요청한다.
6. `show_agent_trace()`로 두 Tool의 호출 순서·인자·호출 ID를 확인한다.

도구를 직접 호출하여 변환 결과를 먼저 검증한 뒤 Agent에 연결한다. 최종 답변만 확인하지 말고 첫 Tool의 온도가 두 번째 Tool의 `temperature` 인자로 전달되었는지, 각 `ToolMessage.tool_call_id`가 해당 `tool_calls.id`와 연결되는지 확인한다.
